# 07. Pairwise and Preference Evaluation

Absolute scores are useful, but many prompt decisions are comparative. This notebook shows how to compare prompt variants head-to-head with pairwise judging, win rates, and uncertainty-aware summaries.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Learning goals

By the end you will be able to:

- set up A versus B prompt comparisons
- compute win rate, tie rate, and uncertainty intervals
- reduce order bias with randomized presentation
- inspect borderline cases instead of trusting only one aggregate number

## Why pairwise evaluation helps

Sometimes it is easier to answer a comparative question than an absolute one. A judge may struggle to assign a perfect 4 versus 5, but still reliably say which of two answers is better for the task.

## Environment setup

This notebook reuses the shared Qwen setup so we can build an open-source pairwise judge.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
import itertools
import re
from IPython.display import Markdown, display

random.seed(19)

def show_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    display(Markdown("\n".join([header, divider] + body)))

def mean(values):
    values = list(values)
    return round(sum(values) / len(values), 3) if values else 0.0

## Step 1: Create a prompt-comparison benchmark

Each case is a tiny project-update task. We know the important facts, so we can generate several prompt-style answers and ask the judge which one is more useful.

In [ ]:
pairwise_cases = [
    {
        "id": "p1",
        "request": "Summarize the mobile release update for the VP in a compact form.",
        "decision": "Delay the mobile launch by one week.",
        "owner": "Maya",
        "deadline": "next Friday",
        "risk": "an Android crash bug is still open",
        "extra": "the web release remains on schedule",
    },
    {
        "id": "p2",
        "request": "Summarize the customer onboarding experiment for leadership.",
        "decision": "Keep the guided setup flow for enterprise accounts.",
        "owner": "Jules",
        "deadline": "ship the rollout on Tuesday",
        "risk": "self-serve accounts still churn after step three",
        "extra": "enterprise completion improved by 14 percent",
    },
    {
        "id": "p3",
        "request": "Summarize the support automation pilot for the weekly review.",
        "decision": "Expand the pilot to the billing queue only.",
        "owner": "Priya",
        "deadline": "review results in two weeks",
        "risk": "refund cases still need manual review",
        "extra": "resolution time dropped by 18 percent",
    },
    {
        "id": "p4",
        "request": "Summarize the search relevance tuning update.",
        "decision": "Keep the new reranker for documentation search.",
        "owner": "Liam",
        "deadline": "monitor metrics through month end",
        "risk": "developer queries still miss older migration guides",
        "extra": "click-through rate improved on the top three results",
    },
    {
        "id": "p5",
        "request": "Summarize the pricing page experiment for the GTM lead.",
        "decision": "Use the simpler plan grid for paid ads traffic.",
        "owner": "Nora",
        "deadline": "launch the split by Wednesday",
        "risk": "the annual-plan tooltip still confuses small teams",
        "extra": "demo requests rose without a signup increase",
    },
    {
        "id": "p6",
        "request": "Summarize the data retention policy rollout.",
        "decision": "Pause full rollout until the export audit is fixed.",
        "owner": "Owen",
        "deadline": "reassess after the Friday audit review",
        "risk": "customers cannot yet verify deleted exports",
        "extra": "legal approved the policy language",
    },
]

show_table(
    [
        {
            "id": case["id"],
            "request": case["request"][:52] + "...",
            "decision": case["decision"],
        }
        for case in pairwise_cases
    ]
)

## Step 2: Define prompt variants

These functions stand in for different prompt styles. The structured variant is disciplined, the baseline is terse but incomplete, and the polished variant sounds good while occasionally smoothing over risks.

In [ ]:
def baseline_answer(case):
    return (
        f"{case['decision']} {case['extra'].capitalize()}. "
        f"{case['owner']} is handling the update."
    )

def structured_answer(case):
    return (
        f"Decision: {case['decision']}\n"
        f"Owner: {case['owner']}\n"
        f"Next milestone: {case['deadline']}\n"
        f"Risk: {case['risk']}\n"
        f"Context: {case['extra']}."
    )

def polished_answer(case):
    optimistic_tail = "The team is in a strong position and no major blockers remain."
    if case["id"] in {"p1", "p6"}:
        optimistic_tail = "Momentum is strong and stakeholders can stay confident."
    return (
        f"{case['decision']} {case['extra'].capitalize()}. "
        f"Plan owner: {case['owner']}. "
        f"Next step is to move toward {case['deadline']}. "
        f"{optimistic_tail}"
    )

VARIANTS = {
    "baseline": baseline_answer,
    "structured": structured_answer,
    "polished": polished_answer,
}

In [ ]:
sample_case = pairwise_cases[0]
for name, fn in VARIANTS.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print(fn(sample_case))

## Step 3: Write the pairwise judging prompt

A pairwise judge should focus on task-relevant criteria, not style alone. Here we ask for A, B, or Tie plus a confidence score and short reasoning.

In [ ]:
PAIRWISE_RUBRIC = """
Prefer the answer that is:
1. more factually faithful to the case details
2. more complete about owner, next step, and risk
3. more directly useful to the stated audience
4. less likely to create false confidence
""".strip()

def build_pairwise_prompt(case, answer_a, answer_b):
    return f"""
Task:
{case["request"]}

Ground truth facts:
- decision: {case["decision"]}
- owner: {case["owner"]}
- deadline: {case["deadline"]}
- risk: {case["risk"]}
- extra context: {case["extra"]}

Answer A:
{answer_a}

Answer B:
{answer_b}

Evaluation rubric:
{PAIRWISE_RUBRIC}

Return JSON only with keys:
winner, confidence, explanation

Rules:
- winner must be A, B, or Tie
- confidence must be an integer from 1 to 5
- explanation should cite concrete differences
- do not use markdown fences
""".strip()

print(build_pairwise_prompt(pairwise_cases[0], baseline_answer(pairwise_cases[0]), structured_answer(pairwise_cases[0])))

## Step 4: Build the pairwise judge wrapper

The wrapper keeps the model interface stable and returns normalized records that the rest of the notebook can aggregate.

In [ ]:
def extract_first_json(text):
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"No JSON object found: {text}")
    return match.group(0)

def normalize_winner(raw_winner):
    winner = str(raw_winner).strip().upper()
    if winner not in {"A", "B", "TIE"}:
        return "Tie"
    return "Tie" if winner == "TIE" else winner

def judge_pair(case, answer_a, answer_b):
    raw_text = generate(
        [
            {"role": "system", "content": "You are a careful pairwise judge. Return JSON only."},
            {"role": "user", "content": build_pairwise_prompt(case, answer_a, answer_b)},
        ],
        max_new_tokens=220,
        temperature=0.0,
        do_sample=False,
    )
    parsed = json.loads(extract_first_json(raw_text))
    return {
        "winner": normalize_winner(parsed.get("winner", "Tie")),
        "confidence": max(1, min(5, int(parsed.get("confidence", 3)))),
        "explanation": str(parsed.get("explanation", "")).strip(),
        "raw_text": raw_text,
    }

## Step 5: Compare one head-to-head pair

We will first compare the baseline prompt against the structured prompt. Order is randomized per case so the left or right slot does not become a hidden bias.

In [ ]:
def compare_variants(left_name, right_name, cases, seed=0):
    rng = random.Random(seed)
    rows = []
    for case in cases:
        swap = rng.random() < 0.5
        if swap:
            answer_a = VARIANTS[right_name](case)
            answer_b = VARIANTS[left_name](case)
            judged = judge_pair(case, answer_a, answer_b)
            winner = left_name if judged["winner"] == "B" else right_name if judged["winner"] == "A" else "tie"
        else:
            answer_a = VARIANTS[left_name](case)
            answer_b = VARIANTS[right_name](case)
            judged = judge_pair(case, answer_a, answer_b)
            winner = left_name if judged["winner"] == "A" else right_name if judged["winner"] == "B" else "tie"

        rows.append(
            {
                "case_id": case["id"],
                "left": left_name,
                "right": right_name,
                "winner": winner,
                "confidence": judged["confidence"],
                "swap_used": swap,
                "explanation": judged["explanation"][:70],
            }
        )
    return rows

baseline_vs_structured = compare_variants("baseline", "structured", pairwise_cases, seed=7)
show_table(baseline_vs_structured, columns=["case_id", "winner", "confidence", "swap_used", "explanation"])

## Step 6: Turn pairwise decisions into win rates

A pairwise summary usually reports wins, losses, ties, and a tie-aware win rate where ties count as half a win.

In [ ]:
def summarize_duel(rows, focus_name):
    wins = sum(1 for row in rows if row["winner"] == focus_name)
    losses = sum(1 for row in rows if row["winner"] not in {focus_name, "tie"})
    ties = sum(1 for row in rows if row["winner"] == "tie")
    total = len(rows)
    tie_aware_rate = (wins + 0.5 * ties) / total if total else 0.0
    return {
        "focus": focus_name,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "tie_aware_win_rate": round(tie_aware_rate, 3),
    }

show_table([summarize_duel(baseline_vs_structured, "structured")])

## Step 7: Add uncertainty with bootstrap intervals

With small datasets, win rates can look more stable than they really are. A simple bootstrap gives a rough uncertainty band without adding external statistics libraries.

In [ ]:
def bootstrap_win_rate(rows, focus_name, trials=1000, seed=0):
    rng = random.Random(seed)
    rates = []
    for _ in range(trials):
        sample = [rng.choice(rows) for _ in rows]
        summary = summarize_duel(sample, focus_name)
        rates.append(summary["tie_aware_win_rate"])
    rates.sort()
    low = rates[int(0.05 * len(rates))]
    high = rates[int(0.95 * len(rates)) - 1]
    return round(low, 3), round(high, 3)

low, high = bootstrap_win_rate(baseline_vs_structured, "structured")
print({"variant": "structured", "bootstrap_90pct_ci": (low, high)})

## Step 8: Run a small round-robin tournament

Pairwise evaluation becomes more useful when you compare a family of prompt variants, not just one duel.

In [ ]:
tournament_rows = []
for left_name, right_name in itertools.combinations(VARIANTS.keys(), 2):
    duel_rows = compare_variants(left_name, right_name, pairwise_cases, seed=13)
    summary = summarize_duel(duel_rows, left_name)
    low, high = bootstrap_win_rate(duel_rows, left_name, seed=13)
    tournament_rows.append(
        {
            "pair": f"{left_name} vs {right_name}",
            "focus": left_name,
            "wins": summary["wins"],
            "losses": summary["losses"],
            "ties": summary["ties"],
            "tie_aware_win_rate": summary["tie_aware_win_rate"],
            "bootstrap_90pct_ci": f"[{low}, {high}]",
        }
    )

show_table(tournament_rows)

## Step 9: Probe order bias explicitly

Randomization helps, but it is healthy to test whether answer order changes outcomes on a few examples.

In [ ]:
bias_rows = []
for case in pairwise_cases[:3]:
    normal = judge_pair(case, baseline_answer(case), structured_answer(case))
    swapped = judge_pair(case, structured_answer(case), baseline_answer(case))
    normalized_normal = "baseline" if normal["winner"] == "A" else "structured" if normal["winner"] == "B" else "tie"
    normalized_swapped = "structured" if swapped["winner"] == "A" else "baseline" if swapped["winner"] == "B" else "tie"
    bias_rows.append(
        {
            "case_id": case["id"],
            "normal_order": normalized_normal,
            "swapped_order": normalized_swapped,
            "same_result": normalized_normal == normalized_swapped,
        }
    )

show_table(bias_rows)

## Step 10: Inspect low-confidence or tied cases

Borderline examples are often more informative than the average. They reveal where your variants differ only slightly or where the task itself is underspecified.

In [ ]:
borderline_rows = [row for row in baseline_vs_structured if row["winner"] == "tie" or row["confidence"] <= 2]
show_table(borderline_rows, columns=["case_id", "winner", "confidence", "explanation"])

## Step 11: Build a preference matrix

A preference matrix makes the round-robin results easier to scan.

In [ ]:
matrix_rows = []
variant_names = list(VARIANTS.keys())
for focus in variant_names:
    row = {"variant": focus}
    for opponent in variant_names:
        if focus == opponent:
            row[opponent] = "—"
            continue
        duel_rows = compare_variants(focus, opponent, pairwise_cases, seed=29)
        row[opponent] = summarize_duel(duel_rows, focus)["tie_aware_win_rate"]
    matrix_rows.append(row)

show_table(matrix_rows, columns=["variant"] + variant_names)

## Useful takeaways

Pairwise evaluation is especially helpful when:

- absolute scores are noisy
- you mainly care about choosing between prompt variants
- you want head-to-head evidence before declaring a winner
- you can inspect ties and low-confidence cases

## Important limits

Pairwise methods still have limits:

- a strong-sounding answer can sometimes beat a better one
- order bias can leak in if you do not randomize
- small datasets create wide uncertainty bands
- pairwise wins do not explain every failure mode by themselves

## Suggested extensions

Good next steps:

1. compare each pair twice with two judge prompts
2. add human adjudication for ties
3. attach cost and latency so you compare quality and efficiency together